# 02 · Medical triage: a three-action decision under asymmetric costs

> **Not a medical device.** Synthetic data, methodology demonstration only. Nothing here is
> validated for clinical use.

The point of this notebook is that **the cost matrix, not the model, decides the operating
point** — and that abstention has to be priced.


In [ ]:
import numpy as np
import pandas as pd
from slm_decision import (DecisionSpec, RulePolicy, CostOptimalPolicy,
                          TemperatureCalibrator, evaluate_policy, ece)

spec = DecisionSpec(
    name="note_triage",
    actions=["routine", "urgent", "escalate"],
    # outcomes are alphabetical: bad = deterioration, good = stable
    costs={("routine", "bad"): 100.0, ("routine", "good"): 0.0,
           ("urgent", "bad"): 8.0, ("urgent", "good"): 12.0,
           ("escalate", "bad"): 4.0, ("escalate", "good"): 4.0},
    abstain="escalate",
    max_abstain_rate=0.30,
    # a clinician reviewing urgently is an acceptable response to a deteriorating
    # patient, even though 'urgent' is never cost-minimal under perfect foresight
    acceptable={"bad": ["escalate", "urgent"], "good": ["routine", "urgent"]},
)
print(spec.outcomes, spec.thresholds())

## Synthetic triage notes and a hashed bag-of-words model

`TfidfVectorizer` + `LogisticRegression` is the reference recipe; this notebook implements the
same idea in numpy (hashed bag-of-words + softmax regression) so it runs with no extra
dependencies. Swap in scikit-learn or a fine-tuned encoder and everything downstream is
unchanged.


In [ ]:
rng = np.random.default_rng(7)

SEVERE = ["crushing chest pain", "shortness of breath at rest", "uncontrolled bleeding",
          "sudden confusion", "fainting episode"]
MILD = ["mild sore throat", "routine blood pressure check", "prescription refill",
        "follow-up on healed sprain", "seasonal allergy"]

def make_note(severe, mislabelled=False):
    """~20% of notes use the other class's vocabulary: realistic text noise."""
    if mislabelled:
        severe = not severe
    words = (SEVERE if severe else MILD)
    extra = ["patient reports", "no fever", "since yesterday", "denies dizziness",
             "history of hypertension", "afebrile"]
    parts = [rng.choice(words)] + list(rng.choice(extra, 3, replace=False))
    rng.shuffle(parts)
    return " , ".join(parts)

n = 3000
severe = rng.random(n) < 0.35
mislabelled = rng.random(n) < 0.20
notes = [make_note(bool(s), bool(m)) for s, m in zip(severe, mislabelled)]
# the spec's outcomes are sorted ['bad', 'good'], so 1 = stable (the "good" outcome)
y = (~severe).astype(int)
print(notes[0], "-> y =", y[0], "(1 = stable)")

In [ ]:
DIM = 2048

def hashed_features(notes, dim=DIM):
    X = np.zeros((len(notes), dim))
    for i, note in enumerate(notes):
        for tok in note.split():
            X[i, hash(tok) % dim] += 1.0
    X = np.log1p(X)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, 1e-9)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def train_logreg(X, y, epochs=400, lr=0.5, l2=1e-3):
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(epochs):
        p = 1 / (1 + np.exp(-(X @ w + b)))
        w -= lr * (X.T @ (p - y) / n + l2 * w)
        b -= lr * (p - y).mean()
    return w, b

split = int(0.7 * n)
X = hashed_features(notes)
w, b = train_logreg(X[:split], y[:split])
p_val = 1 / (1 + np.exp(-(X[split:] @ w + b)))
print("val AUC-ish accuracy at 0.5:", ((p_val > 0.5) == y[split:]).mean().round(3))

## From probabilities to actions

Three policies on the same scores: argmax@0.5 (the naive choice), a fixed rule, and the
cost-optimal policy under a 30% escalation cap.


In [ ]:
y_val = y[split:]
rows = []

naive = RulePolicy(spec, threshold=0.5, positive="urgent", negative="routine")
rows.append({"policy": "argmax@0.5", **evaluate_policy(spec, naive, y_val, probs=p_val).to_dict()})

cost_pol = CostOptimalPolicy(spec, max_abstain_rate=0.30)
rows.append({"policy": "cost-optimal (30% cap)",
             **evaluate_policy(spec, cost_pol, y_val, probs=p_val).to_dict()})

cost_tight = CostOptimalPolicy(spec, max_abstain_rate=0.10)
rows.append({"policy": "cost-optimal (10% cap)",
             **evaluate_policy(spec, cost_tight, y_val, probs=p_val).to_dict()})

print(pd.DataFrame(rows)[["policy", "accuracy", "cost_per_case", "abstain_rate", "coverage", "ece"]].round(3).to_string(index=False))

## Calibration changes the decision

Temperature scaling does not change the ranking, but it changes which cases cross the
threshold — which is where severe misses live.


In [ ]:
cal = TemperatureCalibrator().fit(p_val, y_val)
p_cal = cal.transform(p_val)

for name, p in [("raw", p_val), ("calibrated", p_cal)]:
    rep = evaluate_policy(spec, cost_pol, y_val, probs=p)
    print("%-11s cost/case %6.2f  abstain %.3f  ece %.4f" %
          (name, rep.cost_per_case, rep.abstain_rate, rep.ece))

print("temperature:", round(cal.temperature, 3))

In [ ]:
# severe misses: true deterioration (y == 0, the "bad" outcome) routed to 'routine'
actions = cost_pol.act_batch(p_cal)
severe_idx = np.where(y_val == 0)[0]
misses = np.mean([actions[i] == "routine" for i in severe_idx])
print("severe misses: %.2f%% of true deterioration cases" % (100 * misses))

## Takeaways

- The cost-optimal rule cuts cost per case sharply at the same accuracy — thresholds come
  from costs, not from 0.5.
- The abstain cap is a business constraint: tightening it from 30% to 10% raises cost.
  That trade-off is a capacity conversation, not an ML one.
- Calibration changes the decision, not just the reported confidence.
- Swap the numpy model for a fine-tuned 66M encoder and nothing downstream changes.
